# CMPE 58T — 05a: Morphological Feature Injection V2 (UD FEATS)

Built directly on the working `phase5_morph_injection_ud_feats_final.ipynb`.
Only additions: `PRED_DIR`/`RESULTS_DIR`, checkpoint skip logic, per-relation LAS, npy saving.
All training/evaluation logic is unchanged from the original working notebook.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR  = "/content/drive/MyDrive/58T - Application Project"
DATA_DIR     = os.path.join(PROJECT_DIR, "data")
VOCAB_DIR    = os.path.join(PROJECT_DIR, "vocab")
MODEL_DIR    = os.path.join(PROJECT_DIR, "models")
MORPH_DIR    = os.path.join(MODEL_DIR, "morph")
PRED_DIR     = os.path.join(PROJECT_DIR, "predictions")
RESULTS_DIR  = os.path.join(PROJECT_DIR, "results")

for d in [DATA_DIR, VOCAB_DIR, MODEL_DIR, MORPH_DIR, PRED_DIR, RESULTS_DIR]:
    os.makedirs(d, exist_ok=True)

for tb in ["BOUN", "IMST", "PUD"]:
    path = os.path.join(DATA_DIR, f"UD_Turkish-{tb}")
    print(f"  {'OK' if os.path.isdir(path) else 'MISSING'}  UD_Turkish-{tb}")

feat_vocab_path = os.path.join(VOCAB_DIR, "feat_vocab.json")
print(f"  {'OK' if os.path.exists(feat_vocab_path) else 'MISSING'}  feat_vocab.json")


Mounted at /content/drive
  OK  UD_Turkish-BOUN
  OK  UD_Turkish-IMST
  OK  UD_Turkish-PUD
  OK  feat_vocab.json


In [2]:
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118 -q
!pip install "transformers==4.35.2" -q
!pip install supar -q
!pip install conllu -q

import os

PARSER_PY = "/usr/local/lib/python3.12/dist-packages/supar/parsers/parser.py"
OLD = "state = torch.load(path if os.path.exists(path) else download(supar.MODEL[src].get(path, path), reload=reload))"
NEW = "state = torch.load(path if os.path.exists(path) else download(supar.MODEL[src].get(path, path), reload=reload), weights_only=False)"
with open(PARSER_PY, "r") as f:
    src = f.read()
if OLD in src:
    with open(PARSER_PY, "w") as f:
        f.write(src.replace(OLD, NEW))
    print("Patch 1 applied : torch.load weights_only=False")
else:
    print("Patch 1 OK      : already patched")

from transformers import AdamW, get_linear_schedule_with_warmup
print("Patch 2 OK      : transformers.AdamW available")

import torch, transformers
print(f"\ntransformers : {transformers.__version__}")
print(f"PyTorch      : {torch.__version__}")
print(f"CUDA         : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU          : {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: No GPU — switch runtime before training")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 123.5/123.5 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.9/7.9 MB 160.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 37.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 138.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 5.4.1 requires transformers<6.0.0,>=4.41.0, but you have transformers 4.35.2 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.1/93.1 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 77.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 773.7/773.7 kB 70.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 60.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 418.7/418.7 kB 46.8 MB/

/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(


Patch 2 OK      : transformers.AdamW available

transformers : 4.35.2
PyTorch      : 2.10.0+cu128
CUDA         : True
GPU          : NVIDIA A100-SXM4-40GB


In [3]:
import os, glob, json
import torch

BERTURK_NAME = "dbmdz/bert-base-turkish-uncased"
DEVICE       = "cuda" if torch.cuda.is_available() else "cpu"

MORPH_DIM  = 32
MAX_FEATS  = 16

HIDDEN_SIZE  = 800
DROPOUT      = 0.33
BATCH_SIZE   = 5000
MAX_EPOCHS   = 10
PATIENCE     = 5
LR           = 2e-3
BERT_LR      = 2e-5
WARMUP       = 0.1
WEIGHT_DECAY = 0.01
EPS          = 1e-8
GRAD_CLIP    = 5.0

SUPAR_EXTRA = dict(
    mbr=True, partial=False, tree=False,
    proj=False, punct=False, update_steps=1, clip=GRAD_CLIP
)

def get_splits(tb_name):
    folder = os.path.join(DATA_DIR, f"UD_Turkish-{tb_name}")
    files  = sorted(glob.glob(os.path.join(folder, "*.conllu")))
    splits = {}
    for f in files:
        name = os.path.basename(f)
        if   "train" in name: splits["train"] = f
        elif "dev"   in name: splits["dev"]   = f
        elif "test"  in name: splits["test"]  = f
    return splits

BOUN = get_splits("BOUN")
IMST = get_splits("IMST")
PUD  = get_splits("PUD")

MODEL_V1         = os.path.join(MODEL_DIR,  "baseline_boun.pt")       # from 02_baseline
MODEL_V2         = os.path.join(MORPH_DIR,  "morph_ud_feats_boun.pt")
MORPH_STATE_V2   = MODEL_V2 + ".morph_state.pt"

print(f"Device      : {DEVICE}")
print(f"MODEL_V1    : {MODEL_V1}  {'(exists)' if os.path.exists(MODEL_V1) else '(MISSING)'}")
print(f"MODEL_V2    : {MODEL_V2}  {'(exists)' if os.path.exists(MODEL_V2) else '(new)'}")


Device      : cuda
MODEL_V1    : /content/drive/MyDrive/58T - Application Project/models/baseline_boun.pt  (exists)
MODEL_V2    : /content/drive/MyDrive/58T - Application Project/models/morph/morph_ud_feats_boun.pt  (new)


In [4]:
import json

feat_vocab_path = os.path.join(VOCAB_DIR, "feat_vocab.json")

if not os.path.exists(feat_vocab_path):
    print("feat_vocab.json not found — rebuilding from all treebanks...")
    import conllu
    from collections import Counter
    val_counts = Counter()
    for path in [BOUN["train"], BOUN["dev"], BOUN["test"],
                 IMST["train"], IMST["dev"], IMST["test"], PUD["test"]]:
        with open(path, "r", encoding="utf-8") as f:
            sents = conllu.parse(f.read())
        for sent in sents:
            for tok in sent:
                if isinstance(tok["id"], int) and tok["feats"]:
                    for k, v in tok["feats"].items():
                        val_counts[f"{k}={v}"] += 1
    FEAT_VOCAB = {kv: idx for idx, (kv, _) in enumerate(val_counts.most_common())}
    FEAT_VOCAB["<PAD>"] = len(FEAT_VOCAB)
    FEAT_VOCAB["<UNK>"] = len(FEAT_VOCAB)
    with open(feat_vocab_path, "w", encoding="utf-8") as f:
        json.dump(FEAT_VOCAB, f, ensure_ascii=False, indent=2)
    print(f"Built and saved: {len(FEAT_VOCAB)} entries")
else:
    with open(feat_vocab_path, "r", encoding="utf-8") as f:
        FEAT_VOCAB = json.load(f)
    print(f"Loaded feat_vocab.json: {len(FEAT_VOCAB)} entries")

PAD_IDX    = FEAT_VOCAB["<PAD>"]
UNK_IDX    = FEAT_VOCAB["<UNK>"]
VOCAB_SIZE = len(FEAT_VOCAB)
print(f"  Vocab size : {VOCAB_SIZE}  PAD={PAD_IDX}  UNK={UNK_IDX}")


Loaded feat_vocab.json: 87 entries
  Vocab size : 87  PAD=85  UNK=86


In [5]:
import torch
import torch.nn as nn
import types

class MorphEmbedder(nn.Module):
    def __init__(self, vocab_size, morph_dim, pad_idx):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, morph_dim, padding_idx=pad_idx)
        nn.init.normal_(self.embedding.weight, std=0.01)
        with torch.no_grad():
            self.embedding.weight[pad_idx].fill_(0)
    def forward(self, morph_indices):
        return self.embedding(morph_indices).sum(dim=2)

class MorphProjection(nn.Module):
    def __init__(self, n_hidden, morph_dim):
        super().__init__()
        self.proj = nn.Linear(n_hidden + morph_dim, n_hidden)
        nn.init.eye_(self.proj.weight[:n_hidden, :n_hidden])
        nn.init.zeros_(self.proj.weight[:, n_hidden:])
        nn.init.zeros_(self.proj.bias)
    def forward(self, bert_out, morph_emb):
        return self.proj(torch.cat([bert_out, morph_emb], dim=-1))

FEATS_COL = 5

def batch_to_morph_tensor(batch, feat_vocab, pad_idx, unk_idx, max_feats, device):
    batch_size = len(batch.sentences)
    seq_len = max(len(sent.values[FEATS_COL]) + 1 for sent in batch.sentences)
    morph = torch.full((batch_size, seq_len, max_feats), pad_idx, dtype=torch.long)
    for b, sent in enumerate(batch.sentences):
        feats_col = sent.values[FEATS_COL]
        for t, feat_str in enumerate(feats_col, start=1):
            if feat_str == "_" or not feat_str:
                continue
            pairs = feat_str.split("|")
            for f_idx, pair in enumerate(pairs[:max_feats]):
                morph[b, t, f_idx] = feat_vocab.get(pair.strip(), unk_idx)
    return morph.to(device)

def patch_model_forward(parser, morph_embedder, morph_projection):
    from supar.utils.common import MIN
    _embedder    = morph_embedder
    _projection  = morph_projection
    _orig_encode = parser.model.encode

    def forward_with_morph(self, words, feats=None, morph_indices=None):
        x = _orig_encode(words, feats)
        if morph_indices is not None:
            morph_emb = _embedder(morph_indices)
            x = _projection(x, morph_emb)
        x = self.encoder_dropout(x)
        mask  = words.ne(self.args.pad_index) if len(words.shape) < 3 else words.ne(self.args.pad_index).any(-1)
        arc_d = self.arc_mlp_d(x);  arc_h = self.arc_mlp_h(x)
        sib_s = self.sib_mlp_s(x);  sib_d = self.sib_mlp_d(x);  sib_h = self.sib_mlp_h(x)
        rel_d = self.rel_mlp_d(x);  rel_h = self.rel_mlp_h(x)
        s_arc = self.arc_attn(arc_d, arc_h).masked_fill_(~mask.unsqueeze(1), MIN)
        s_sib = self.sib_attn(sib_s, sib_d, sib_h).permute(0, 3, 1, 2)
        s_rel = self.rel_attn(rel_d, rel_h).permute(0, 2, 3, 1)
        return s_arc, s_sib, s_rel

    parser.model.forward = types.MethodType(forward_with_morph, parser.model)
    print("Model forward patched.")

print("Morph modules defined.")


Morph modules defined.


In [6]:
from supar import CRF2oDependencyParser

if os.path.exists(MODEL_V2):
    print(f"V2 checkpoint found — loading from {MODEL_V2}")
    parser = CRF2oDependencyParser.load(MODEL_V2)
else:
    if os.path.exists(MODEL_V1):
        print(f"Loading baseline_boun.pt as starting point: {MODEL_V1}")
        parser = CRF2oDependencyParser.load(MODEL_V1)
    else:
        raise FileNotFoundError(f"MODEL_V1 not found: {MODEL_V1}. Run 02_baseline.ipynb first.")

parser.args.update(SUPAR_EXTRA)
parser.model = parser.model.to(DEVICE)
n_hidden = parser.model.encoder.n_out

morph_embedder   = MorphEmbedder(VOCAB_SIZE, MORPH_DIM, PAD_IDX).to(DEVICE)
morph_projection = MorphProjection(n_hidden, MORPH_DIM).to(DEVICE)

# Load saved morph weights if resuming from V2 checkpoint
if os.path.exists(MORPH_STATE_V2):
    state = torch.load(MORPH_STATE_V2, map_location=DEVICE, weights_only=False)
    morph_embedder.load_state_dict(state["morph_embedder"])
    morph_projection.load_state_dict(state["morph_projection"])
    print("Morph module weights loaded from checkpoint.")

patch_model_forward(parser, morph_embedder, morph_projection)

morph_params = (sum(p.numel() for p in morph_embedder.parameters()) +
                sum(p.numel() for p in morph_projection.parameters()))
print(f"  n_hidden={n_hidden}  morph_params={morph_params:,}  device={DEVICE}")


Loading baseline_boun.pt as starting point: /content/drive/MyDrive/58T - Application Project/models/baseline_boun.pt


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/59.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


pytorch_model.bin:   0%|          | 0.00/445M [00:00<?, ?B/s]

Model forward patched.
  n_hidden=768  morph_params=617,952  device=cuda


In [7]:
from supar.utils import Dataset
from transformers import AdamW, get_linear_schedule_with_warmup

def make_loader(parser, path, batch_size, shuffle=True):
    dataset = Dataset(parser.transform, path)
    dataset.build(batch_size, 32)
    return dataset.loader

print("Building data loaders...")
train_loader = make_loader(parser, BOUN["train"], BATCH_SIZE, shuffle=True)
dev_loader   = make_loader(parser, BOUN["dev"],   BATCH_SIZE, shuffle=False)
test_loader  = make_loader(parser, BOUN["test"],  BATCH_SIZE, shuffle=False)
pud_loader   = make_loader(parser, PUD["test"],   BATCH_SIZE, shuffle=False)
print(f"  train={len(train_loader)}  dev={len(dev_loader)}  test={len(test_loader)}  pud={len(pud_loader)} batches")

total_steps  = len(train_loader) * MAX_EPOCHS
warmup_steps = int(total_steps * WARMUP)

all_params  = (list(parser.model.parameters()) +
               list(morph_embedder.parameters()) +
               list(morph_projection.parameters()))
bert_params = list(parser.model.encoder.parameters())
task_params = ([p for p in parser.model.parameters()
                if not any(p is bp for bp in bert_params)] +
               list(morph_embedder.parameters()) +
               list(morph_projection.parameters()))

optimizer = AdamW([{"params": bert_params, "lr": BERT_LR},
                   {"params": task_params, "lr": LR}],
                  weight_decay=WEIGHT_DECAY, eps=EPS)
scheduler = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)
print(f"  total_steps={total_steps}  warmup={warmup_steps}")


Building data loaders...


/usr/local/lib/python3.12/dist-packages/supar/utils/fn.py:155: UserWarning: Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will result either in an error or a different result (Triggered internally at /pytorch/torch/csrc/autograd/python_variable_indexing.cpp:347.)
  out_tensor[i][[slice(-i, None) if padding_side == 'left' else slice(0, i) for i in tensor.size()]] = tensor


  train=40  dev=32  test=32  pud=32 batches


/usr/local/lib/python3.12/dist-packages/transformers/optimization.py:411: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


  total_steps=400  warmup=40


In [8]:
from supar.utils.metric import AttachmentMetric

def run_epoch(parser, loader, morph_embedder, morph_projection,
              optimizer, scheduler, device, use_morph, training=True):
    if training:
        parser.model.train(); morph_embedder.train(); morph_projection.train()
    else:
        parser.model.eval(); morph_embedder.eval(); morph_projection.eval()

    total_loss = 0.0
    metric     = AttachmentMetric()
    ctx = torch.enable_grad() if training else torch.no_grad()

    with ctx:
        for batch in loader:
            words, texts, *feats, arcs, sibs, rels = batch
            word_mask = words.ne(parser.args.pad_index)
            mask      = word_mask if len(words.shape) < 3 else word_mask.any(-1)
            mask[:, 0] = 0
            words = words.to(device)
            feats = [f.to(device) for f in feats]
            arcs  = arcs.to(device);  sibs = sibs.to(device)
            rels  = rels.to(device);  mask = mask.to(device)

            morph_indices = None
            if use_morph:
                morph_indices = batch_to_morph_tensor(
                    batch, FEAT_VOCAB, PAD_IDX, UNK_IDX, MAX_FEATS, device)
                morph_indices = morph_indices[:, :words.shape[1], :]

            s_arc, s_sib, s_rel = parser.model(words, feats, morph_indices)
            loss, s_arc = parser.model.loss(
                s_arc, s_sib, s_rel, arcs, sibs, rels, mask,
                parser.args.mbr, parser.args.partial)

            if training:
                optimizer.zero_grad(); loss.backward()
                import torch.nn as nn
                nn.utils.clip_grad_norm_(all_params, GRAD_CLIP)
                optimizer.step(); scheduler.step()

            arc_preds, rel_preds = parser.model.decode(
                s_arc, s_sib, s_rel, mask,
                parser.args.tree, parser.args.mbr, parser.args.proj)
            total_loss += loss.item()
            metric(arc_preds, rel_preds, arcs, rels, mask)

    return total_loss / len(loader), metric


if os.path.exists(MODEL_V2) and os.path.exists(MORPH_STATE_V2):
    print("V2 checkpoint exists — skipping training.")
    print("Delete morph_ud_feats_boun.pt and .morph_state.pt to retrain.")
else:
    print(f"Training V2 | Epochs={MAX_EPOCHS}  Patience={PATIENCE}  Device={DEVICE}")
    print("=" * 65)

    best_las = 0.0; patience_ct = 0; history = []

    for epoch in range(1, MAX_EPOCHS + 1):
        train_loss, _ = run_epoch(parser, train_loader, morph_embedder, morph_projection,
                                   optimizer, scheduler, DEVICE, use_morph=True, training=True)
        dev_loss, dev_m = run_epoch(parser, dev_loader, morph_embedder, morph_projection,
                                     optimizer, scheduler, DEVICE, use_morph=True, training=False)
        uas = dev_m.uas * 100; las = dev_m.las * 100
        print(f"Epoch {epoch:>2}/{MAX_EPOCHS}  train {train_loss:.4f} | dev {dev_loss:.4f} | UAS {uas:.2f} LAS {las:.2f}", end="")

        if las > best_las:
            best_las = las; patience_ct = 0
            torch.save({"morph_embedder": morph_embedder.state_dict(),
                        "morph_projection": morph_projection.state_dict()},
                       MORPH_STATE_V2)
            parser.save(MODEL_V2)
            print(" (saved)", end="")
        else:
            patience_ct += 1
            print(f" (patience {patience_ct}/{PATIENCE})", end="")
        print()
        history.append({"epoch": epoch, "train_loss": train_loss,
                        "dev_loss": dev_loss, "dev_uas": uas, "dev_las": las})
        if patience_ct >= PATIENCE:
            print(f"Early stopping at epoch {epoch}."); break

    print(f"\nBest dev LAS: {best_las:.2f}")
    with open(os.path.join(RESULTS_DIR, "morph_v2_history.json"), "w") as f:
        json.dump(history, f, indent=2)
    print("History saved.")


Training V2 | Epochs=10  Patience=5  Device=cuda


/usr/local/lib/python3.12/dist-packages/supar/utils/fn.py:155: UserWarning: Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will result either in an error or a different result (Triggered internally at /pytorch/torch/csrc/autograd/python_variable_indexing.cpp:347.)
  out_tensor[i][[slice(-i, None) if padding_side == 'left' else slice(0, i) for i in tensor.size()]] = tensor


Epoch  1/10  train 208.5319 | dev 81.2203 | UAS 54.85 LAS 30.42 (saved)


/usr/local/lib/python3.12/dist-packages/supar/utils/fn.py:155: UserWarning: Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will result either in an error or a different result (Triggered internally at /pytorch/torch/csrc/autograd/python_variable_indexing.cpp:347.)
  out_tensor[i][[slice(-i, None) if padding_side == 'left' else slice(0, i) for i in tensor.size()]] = tensor


Epoch  2/10  train 10.7017 | dev 1.9721 | UAS 70.94 LAS 51.35 (saved)


...

Epoch  8/10  train 0.6858 | dev 1.8124 | UAS 80.01 LAS 71.68 (saved)


/usr/local/lib/python3.12/dist-packages/supar/utils/fn.py:155: UserWarning: Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will result either in an error or a different result (Triggered internally at /pytorch/torch/csrc/autograd/python_variable_indexing.cpp:347.)
  out_tensor[i][[slice(-i, None) if padding_side == 'left' else slice(0, i) for i in tensor.size()]] = tensor


Epoch  9/10  train 0.6380 | dev 1.7480 | UAS 80.22 LAS 71.94 (saved)


/usr/local/lib/python3.12/dist-packages/supar/utils/fn.py:155: UserWarning: Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will result either in an error or a different result (Triggered internally at /pytorch/torch/csrc/autograd/python_variable_indexing.cpp:347.)
  out_tensor[i][[slice(-i, None) if padding_side == 'left' else slice(0, i) for i in tensor.size()]] = tensor


Epoch 10/10  train 0.5955 | dev 1.8170 | UAS 80.36 LAS 72.31 (saved)

Best dev LAS: 72.31
History saved.


In [9]:
import json, numpy as np
import conllu as conllu_lib
from collections import defaultdict

# Load best morph weights
if os.path.exists(MORPH_STATE_V2):
    state = torch.load(MORPH_STATE_V2, map_location=DEVICE, weights_only=False)
    morph_embedder.load_state_dict(state["morph_embedder"])
    morph_projection.load_state_dict(state["morph_projection"])
    print("Best morph weights loaded.")

def evaluate_loader(parser, loader, morph_embedder, morph_projection,
                    device, use_morph, label):
    _, metric = run_epoch(parser, loader, morph_embedder, morph_projection,
                           None, None, device, use_morph=use_morph, training=False)
    result = {"system": label,
              "UAS": round(metric.uas * 100, 2),
              "LAS": round(metric.las * 100, 2)}
    print(f"  {label:<45} UAS {result['UAS']:>6.2f}   LAS {result['LAS']:>6.2f}")
    return result

def per_relation_las(gold_sents, pred_sents):
    counts = defaultdict(lambda: [0, 0])
    for gold, pred in zip(gold_sents, pred_sents):
        gt = [t for t in gold if isinstance(t["id"], int)]
        pt = [t for t in pred if isinstance(t["id"], int)]
        for g, p in zip(gt, pt):
            counts[g["deprel"]][1] += 1
            if g["head"] == p["head"] and g["deprel"] == p["deprel"]:
                counts[g["deprel"]][0] += 1
    return {rel: {"correct": c, "total": t,
                  "LAS": round(100 * c / t, 2) if t else 0.0}
            for rel, (c, t) in sorted(counts.items(), key=lambda x: -x[1][1])}

results = []
print("=" * 65)

# V1: load baseline_boun.pt, patch forward, build fresh loaders, evaluate
parser_v1 = CRF2oDependencyParser.load(MODEL_V1)
parser_v1.args.update(SUPAR_EXTRA)
parser_v1.model = parser_v1.model.to(DEVICE)
patch_model_forward(parser_v1, morph_embedder, morph_projection)
test_loader_v1 = make_loader(parser_v1, BOUN["test"], BATCH_SIZE, shuffle=False)
pud_loader_v1  = make_loader(parser_v1, PUD["test"],  BATCH_SIZE, shuffle=False)

r = evaluate_loader(parser_v1, test_loader_v1, morph_embedder, morph_projection,
                    DEVICE, use_morph=False, label="V1: BERTurk only (baseline)")
r["test"] = "BOUN"; results.append(r)
r = evaluate_loader(parser_v1, pud_loader_v1, morph_embedder, morph_projection,
                    DEVICE, use_morph=False, label="V1: BERTurk only -> PUD")
r["test"] = "PUD"; results.append(r)

# V2: use the same test_loader and pud_loader built in cell 6
r = evaluate_loader(parser, test_loader, morph_embedder, morph_projection,
                    DEVICE, use_morph=True, label="V2: BERTurk + UD FEATS")
r["test"] = "BOUN"; results.append(r)
r = evaluate_loader(parser, pud_loader, morph_embedder, morph_projection,
                    DEVICE, use_morph=True, label="V2: BERTurk + UD FEATS -> PUD")
r["test"] = "PUD"; results.append(r)

# Save per-sentence npy for McNemar's tests (use parser.predict for conllu)
for sys_name, test_name, conllu_path in [
        ("morph_v2_ud_feats", "BOUN", BOUN["test"]),
        ("morph_v2_ud_feats", "PUD",  PUD["test"])]:
    pred_conllu = os.path.join(PRED_DIR, f"{sys_name}_{test_name}.conllu")
    parser.predict(conllu_path, pred=pred_conllu, verbose=False)
    with open(conllu_path, encoding="utf-8") as f:
        gold_sents = conllu_lib.parse(f.read())
    with open(pred_conllu, encoding="utf-8") as f:
        pred_sents = conllu_lib.parse(f.read())
    sent_las = [
        sum(1 for g, p in zip(
            [t for t in gold if isinstance(t["id"], int)],
            [t for t in pred if isinstance(t["id"], int)])
            if g["head"] == p["head"] and g["deprel"] == p["deprel"]) ==
        len([t for t in gold if isinstance(t["id"], int)])
        for gold, pred in zip(gold_sents, pred_sents)
    ]
    np.save(os.path.join(PRED_DIR, f"{sys_name}_{test_name}_sent_las.npy"),
            np.array(sent_las, dtype=bool))
    print(f"  Saved: {sys_name}_{test_name}.conllu + _sent_las.npy")

# Per-relation LAS on BOUN test
pred_boun = os.path.join(PRED_DIR, "morph_v2_ud_feats_BOUN.conllu")
if os.path.exists(pred_boun):
    with open(BOUN["test"], encoding="utf-8") as f:
        gold_boun = conllu_lib.parse(f.read())
    with open(pred_boun, encoding="utf-8") as f:
        pred_boun_sents = conllu_lib.parse(f.read())
    breakdown = per_relation_las(gold_boun, pred_boun_sents)
    br_path = os.path.join(RESULTS_DIR, "morph_v2_BOUN_per_rel.json")
    with open(br_path, "w") as f:
        json.dump(breakdown, f, indent=2)
    print(f"  Per-relation saved: {br_path}")

# Save results JSON
results_path = os.path.join(RESULTS_DIR, "morph_v2_results.json")
with open(results_path, "w") as f:
    json.dump(results, f, indent=2)
print(f"\nResults saved: {results_path}")


Best morph weights loaded.


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Model forward patched.


/usr/local/lib/python3.12/dist-packages/supar/utils/fn.py:155: UserWarning: Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will result either in an error or a different result (Triggered internally at /pytorch/torch/csrc/autograd/python_variable_indexing.cpp:347.)
  out_tensor[i][[slice(-i, None) if padding_side == 'left' else slice(0, i) for i in tensor.size()]] = tensor


  V1: BERTurk only (baseline)                   UAS  76.63   LAS  63.91
  V1: BERTurk only -> PUD                       UAS  73.71   LAS  59.86
  V2: BERTurk + UD FEATS                        UAS  79.92   LAS  71.79
  V2: BERTurk + UD FEATS -> PUD                 UAS  76.43   LAS  64.05
  Saved: morph_v2_ud_feats_BOUN.conllu + _sent_las.npy
  Saved: morph_v2_ud_feats_PUD.conllu + _sent_las.npy
  Per-relation saved: /content/drive/MyDrive/58T - Application Project/results/morph_v2_BOUN_per_rel.json

Results saved: /content/drive/MyDrive/58T - Application Project/results/morph_v2_results.json


In [10]:
print()
print(f"{'System':<47} {'Test':<6} {'UAS':>7} {'LAS':>7}")
print("-" * 68)
for r in results:
    print(f"  {r['system']:<45} {r['test']:<6} {r['UAS']:>7.2f} {r['LAS']:>7.2f}")

v1_boun = next((r for r in results if "V1" in r["system"] and r["test"] == "BOUN"), None)
v2_boun = next((r for r in results if "V2" in r["system"] and r["test"] == "BOUN"), None)
if v1_boun and v2_boun:
    d_uas = v2_boun["UAS"] - v1_boun["UAS"]
    d_las = v2_boun["LAS"] - v1_boun["LAS"]
    print(f"\nDelta V2 - V1 (BOUN): ΔUAS {d_uas:+.2f}  ΔLAS {d_las:+.2f}")



System                                          Test       UAS     LAS
--------------------------------------------------------------------
  V1: BERTurk only (baseline)                   BOUN     76.63   63.91
  V1: BERTurk only -> PUD                       PUD      73.71   59.86
  V2: BERTurk + UD FEATS                        BOUN     79.92   71.79
  V2: BERTurk + UD FEATS -> PUD                 PUD      76.43   64.05

Delta V2 - V1 (BOUN): ΔUAS +3.29  ΔLAS +7.88


In [11]:
expected = [
    (MORPH_DIR,   "morph_ud_feats_boun.pt"),
    (MORPH_DIR,   "morph_ud_feats_boun.pt.morph_state.pt"),
    (PRED_DIR,    "morph_v2_ud_feats_BOUN.conllu"),
    (PRED_DIR,    "morph_v2_ud_feats_BOUN_sent_las.npy"),
    (PRED_DIR,    "morph_v2_ud_feats_PUD.conllu"),
    (PRED_DIR,    "morph_v2_ud_feats_PUD_sent_las.npy"),
    (RESULTS_DIR, "morph_v2_results.json"),
    (RESULTS_DIR, "morph_v2_history.json"),
    (RESULTS_DIR, "morph_v2_BOUN_per_rel.json"),
]

print(f"{'Status':<8} {'Size':>12}  {'Dir':<14} File")
print("-" * 80)
all_ok = True
for directory, filename in expected:
    path   = os.path.join(directory, filename)
    exists = os.path.exists(path)
    size   = f"{os.path.getsize(path):,} B" if exists else "-"
    status = "OK" if exists else "MISSING"
    folder = os.path.basename(directory)
    if not exists: all_ok = False
    print(f"  {status:<6} {size:>12}  {folder:<14} {filename}")
print()
if all_ok:
    print("All files present. 05a complete.")
else:
    print("Some files missing — rerun the relevant cells.")


Status           Size  Dir            File
--------------------------------------------------------------------------------
  OK     457,211,605 B  morph          morph_ud_feats_boun.pt
  OK      2,474,452 B  morph          morph_ud_feats_boun.pt.morph_state.pt
  OK        981,012 B  predictions    morph_v2_ud_feats_BOUN.conllu
  OK          1,107 B  predictions    morph_v2_ud_feats_BOUN_sent_las.npy
  OK      1,408,758 B  predictions    morph_v2_ud_feats_PUD.conllu
  OK          1,128 B  predictions    morph_v2_ud_feats_PUD_sent_las.npy
  OK            429 B  results        morph_v2_results.json
  OK          1,653 B  results        morph_v2_history.json
  OK          3,099 B  results        morph_v2_BOUN_per_rel.json

All files present. 05a complete.
